In [4]:
import csv #importera csv
from datetime import datetime #Verktyg för tidsstämplig
import json #importera Json strukturerad hirarkisk
import requests #importerar bibliotek för att skicka anrop och hämta priser från coingecko
import time # Import the time module for sleep

class BaseTracker: #Skapar basklass

  def __init__(self, currency="usd"): #konstruktorn standardvalutan är USD
    self.currency = currency.lower() #Sparar valutan i objektet, samt små bokstäver
    self.created_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S") #Hämtar aktuellt datum och klocklas samt sparar som string

class BitcoinTracker(BaseTracker): #Skapar barnklass som ärver från Basetracker

  API_URL = "https://api.coingecko.com/api/v3/simple/price" #Skapar en variabel som sparar Coingeckos API adress

  def __init__(self, currency="usd"): #Startknappen som kör igång
    super().__init__(currency=currency) #anropar föräldraklass och vi får med oss self.currency
    self.headers = {"accept": "application/json", "User-Agent": "Mozilla/5.0"} #Skickar med infornation i vårt API-anrop så att Coingeckos server ser vårt antop
    self.history = []  # Lista för att spara historiska mätningar

  def fetch_and_save(self): # funktionen hämtar priset från API, Validerar och sparar ned till fil

    params = { #Skapar ett Python dictonary med parametrar
        "ids": "bitcoin", #talar om för coingecko att vi vill ha Bitcoin data
        "vs_currencies": f"{self.currency},sek", #Anger vilka valutor vi vill jämföra emot
        "include_24hr_change": "true" #vi vill ha kursföändring i % senaste 24 timmarna
    }
    print(" Hämtar Bitcoin priset från Coingecko")

    try: #Startar en felhantering
      res = requests.get(
          self.API_URL, params=params, headers=self.headers, timeout=10
     )
      res.raise_for_status() #skickar ett fel om http koden är fel


      #Hämtar och kollar datan
      data = res.json()["bitcoin"]
      price_usd = float(data[self.currency])
      price_sek = float(data["sek"])
      change_24h = round(float(data[f"{self.currency}_24h_change"]), 2)
      now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

      #Validering av priset om priset är mindre eller likamed 0 ogiltigt pris
      if price_usd <= 0:
        print("ogiltigt pris")
        return None

      entry = { #skapar en ordbok med all insamlad data i mätningen
          "timestamp": now,
          "crypto": "BITCOIN",
          "price_usd": price_usd,
          "price_sek": price_sek,
          "change_24h_percent": change_24h,
      }
      print(f" Pris: ${price_usd:,} USD ({price_sek:,} SEK)") #Skriver ut priset i terminalen
      return entry

    except requests.exceptions.RequestException as e:
      print(f"❌ Fel vid anrop: {e}")
      return None

  def start_tracking(self, count=3, delay_seconds=5):
    """Loopar och samlar in mätningar i historiklistan, sparar sedan allt."""
    print(f"🔄 Startar automatisk bevakning ({count} mätningar med {delay_seconds}s intervall)...")

    # FOR-LOOP: Körs 'count' antal gånger
    for i in range(1, count + 1):
        print(f"\n--- Mätning {i} av {count} ---")
        entry = self.fetch_and_save()

        if entry:
            self.history.append(entry)  # Lägg till i listan

        # Pausa mellan mätningarna (om det inte är sista varvet)
        if i < count:
            time.sleep(delay_seconds)

    # Spara hela historiklistan till fil när loopen är klar
    if self.history:
        self.save_to_files()

  def save_to_files(self):
    """Sparar alla objekt i historiklistan till JSON och CSV."""

    #sparar till Json med GDPR
    json_data = {
        "source": "CoinGecko API",
        "gdpr_compliant": True,
        "lawful_basis": "Public market financial data (no PII)",
        "total_records": len(self.history),
        "data": self.history,
    }
    with open("bitcoin_data.json", "w", encoding="utf-8") as f: #skapar/Öppnar json filen bitcoin_data.json. W=write,  With stänger därefter
        json.dump(json_data, f, indent=4, ensure_ascii=False)

    #Sparar till CSV
    with open("bitcoin_data.csv", "w", newline="", encoding="utf-8") as f: #Newlineförhindrar att de skapas tomma rader,
        writer = csv.DictWriter(f, fieldnames=list(self.history[0].keys()))
        writer.writeheader()
        writer.writerows(self.history)  # Skriver alla rader i listan

    print(f"\n💾 Sparat {len(self.history)} mätningar i 'bitcoin_data.json' och 'bitcoin_data.csv'!")


#Kör programmet
if __name__ == "__main__": #ser till att koden nedan körs när du startar filen 
  tracker = BitcoinTracker() #skapar ditt tracker object och anropas init
  tracker.start_tracking(count=5, delay_seconds=10)

🔄 Startar automatisk bevakning (5 mätningar med 10s intervall)...

--- Mätning 1 av 5 ---
 Hämtar Bitcoin priset från Coingecko
 Pris: $82,864.0 USD (824,797.0 SEK)

--- Mätning 2 av 5 ---
 Hämtar Bitcoin priset från Coingecko
 Pris: $82,864.0 USD (824,797.0 SEK)

--- Mätning 3 av 5 ---
 Hämtar Bitcoin priset från Coingecko
 Pris: $82,864.0 USD (824,797.0 SEK)

--- Mätning 4 av 5 ---
 Hämtar Bitcoin priset från Coingecko
 Pris: $82,864.0 USD (824,797.0 SEK)

--- Mätning 5 av 5 ---
 Hämtar Bitcoin priset från Coingecko
 Pris: $82,864.0 USD (824,797.0 SEK)

💾 Sparat 5 mätningar i 'bitcoin_data.json' och 'bitcoin_data.csv'!


In [3]:
%%writefile .gitignore
# Python / Colab tillfälliga filer
__pycache__/
*.py[cod]
*$py.class
.ipynb_checkpoints/

# Exporterad data och grafer
bitcoin_data.json
bitcoin_data.csv
bitcoin_30days.png
bitcoin_trend.png

# Miljö- och IDE-inställningar
.env
.venv/
env/
venv/
.vscode/
.DS_Store

Writing .gitignore


In [2]:
%%writefile README.md
# 🪙 Bitcoin Price Tracker

Ett objektorienterat Python-skript som hämtar realtidsdata för Bitcoin-priset via CoinGecko REST API, validerar datan, samlar in historik över tid och sparar mätningarna till strukturerade JSON- och CSV-filer.

## 🚀 Funktioner & Nyckelkoncept
* **Objektorienterad programmering (OOP):** Använder en basklass (`BaseTracker`) och en underklass (`BitcoinTracker`) med arv och `super().__init__()`.
* **API-anrop & Externa bibliotek:** Hämtar levande marknadsdata via `requests` API-anrop.
* **Felhantering & Validering:** Skyddat mot nätverksfel med `try/except` och `raise_for_status()`, samt validerar att mottagna priser är giltiga.
* **Automation & Loopar:** Samlar in flera mätningar automatiskt i en `for`-loop med pausintervall (`time.sleep`).
* **Filhantering:** Exporterar automatiskt historiklistan till både **JSON** och **CSV**.

## 🛠️ Installation & Användning

1. **Klona projektet:**
   ```bash
   git clone <https://github.com/Shorka84/Bitcoin_tracker.git>
   cd bitcoin-tracker

Installera beroenden: pip install requests

Kör programmet: python main.py


📊 Datastruktur & GDPR-efterlevnad
Projektet genererar två filer med insamlad historik:
bitcoin_data.json: Innehåller fullstäniga mätningar tillsammans med GDPR-metadata som verifierar att endast offentlig marknadsdata utan personuppgifter (PII) behandlas.
bitcoin_data.csv: En platt tabellstruktur med kolumnrubriker redo att öppnas i kalkylprogram som Excel.

Writing README.md


In [ ]:
!python main.py